In [21]:
from enum import IntEnum
from aiomoex import get_board_candles
import asyncio
import aiohttp
from datetime import datetime
import pandas as pd


class IntervalEnum(IntEnum):
    MINUTE = 1
    TEN_MINUTES = 10
    HOUR = 60
    DAY = 24
    WEEK = 7
    MONTH = 31

# объявим аннотацию для удобства
StockData = list[dict[str, str | int | float]]

async def fetch_ticker_data(session: aiohttp.ClientSession, ticker: str, interval: IntervalEnum, start_date: str, end_date: str) -> dict[str, StockData]:
    '''Функция получает данные о торгах по заданному тикеру с *start_date* по *end_date* с интервалом *interval*, возвращая словарь, где ключом является тикер, а значением - данные

    Args:
        session (aiohttp.ClientSession): aiottp сессия для отпаравки запросов
        ticker (str): Имя тикера
        interval (IntervalEnum): Одно из доступных значений для интервала времени
        start_date (str): Начальная дата в формате yyyy-mm-dd
        end_date (str): Конечная дата в формате yyyy-mm-dd

    Returns:
        dict[str, list[dict[str, str | int | float]]]: Словарь, где ключ - тикер, а значение - данные, например {'SBER': sber_data}
    '''
    try:
        # получаем данные по переданному тикеру за указанный период
        res = await get_board_candles(session, ticker, interval, start_date, end_date)
        if res:
            df = pd.DataFrame(res)
            
            # Убираем колонку 'end' если она существует
            if 'end' in df.columns:
                df = df.drop('end', axis=1)
            
            # Оставляем только дату (без времени), если это дневные данные или выше
            if interval in [24, 7, 31]:
                df['begin'] = pd.to_datetime(df['begin']).dt.date
            
            # Перемещаем дату в первую колонку
            if 'begin' in df.columns:
                cols = ['begin'] + [col for col in df.columns if col != 'begin']
                df = df[cols]
            
            df['ticker'] = ticker
            return {ticker: df}
        else:
            return {ticker: pd.DataFrame()}
    except Exception as e:
        print(f'Ошибка парсинга. Не удалось получить данные для {ticker}, {e}')
        return {ticker: pd.DataFrame()}

async def get_moex_data(tickers: list[str], start_date: datetime, end_date: datetime = datetime.now(), interval: IntervalEnum = IntervalEnum.DAY) -> dict[str, StockData]:

    if interval not in IntervalEnum:
        raise ValueError(f"Неверный интервал. Допустимые значения: {IntervalEnum}")
    
    end_date_formatted = end_date.strftime('%Y-%m-%d')
    start_date_formatted = start_date.strftime('%Y-%m-%d')

    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(connect=5)) as session:
        # собираем корутины в список
        coros = [fetch_ticker_data(session, ticker, interval, start_date_formatted, end_date_formatted) for ticker in tickers]

        # 'собираем' результаты корутин - непосредственно парсинг
        stock_data = await asyncio.gather(*coros)

    # разворачиваем список словарей в один словарь, например: [ {'SBER': sber_data}, {'GAZP': gazp_data} ] -> { 'SBER': sber_data, 'GAZP': gazp_data }
    stock_data = {ticker: data for element in stock_data for ticker, data in element.items()}
    return stock_data


tickers = ['SBER']

delta = timedelta(days=180)
start_date = datetime.now() - delta

stock_data = await get_moex_data(tickers, start_date=start_date, interval=IntervalEnum.MINUTE)

sber_df = pd.DataFrame(stock_data['SBER'])
sber_df['ticker'] = 'SBER'
sber_df

,open,close,high,low,value,volume,begin,end,ticker
0,298.00,298.00,298.00,298.00,6082180.00,20410,2025-04-16 06:59:00,2025-04-16 06:59:59,SBER
1,298.09,297.65,298.31,297.53,9963145.60,33460,2025-04-16 07:00:00,2025-04-16 07:00:59,SBER
2,297.70,297.89,297.93,297.63,12460925.00,41850,2025-04-16 07:01:00,2025-04-16 07:01:59,SBER
3,297.89,297.91,297.93,297.67,6074049.80,20400,2025-04-16 07:02:00,2025-04-16 07:02:59,SBER
4,297.77,297.90,297.91,297.73,4240380.40,14240,2025-04-16 07:03:00,2025-04-16 07:03:59,SBER
...,...,...,...,...,...,...,...,...,...
146204,285.13,285.56,285.78,285.11,28976491.07,101548,2025-10-13 21:54:00,2025-10-13 21:54:59,SBER
146205,285.57,285.35,285.57,285.32,1727549.35,6050,2025-10-13 21:55:00,2025-10-13 21:55:59,SBER
146206,285.32,285.29,285.32,285.27,97570.42,342,2025-10-13 21:56:00,2025-10-13 21:56:59,SBER
146207,285.30,285.30,285.30,285.29,95289.17,334,2025-10-13 21:57:00,2025-10-13 21:57:59,SBER


In [9]:
gazp_df = pd.DataFrame(stock_data['GAZP'])
gazp_df

,open,close,high,low,value,volume,begin,end
0,131.50,131.50,131.50,131.50,6288330.0,47820,2024-10-14 09:59:00,2024-10-14 09:59:59
1,131.50,130.90,131.50,130.80,95002479.6,724170,2024-10-14 10:00:00,2024-10-14 10:00:59
2,130.91,131.07,131.07,130.67,73470428.9,561470,2024-10-14 10:01:00,2024-10-14 10:01:59
3,131.07,131.11,131.25,131.05,25926925.2,197710,2024-10-14 10:02:00,2024-10-14 10:02:59
4,131.11,131.21,131.27,131.10,11462552.7,87380,2024-10-14 10:03:00,2024-10-14 10:03:59
...,...,...,...,...,...,...,...,...
266069,116.00,116.03,116.04,116.00,104423.3,900,2025-10-13 21:35:00,2025-10-13 21:35:59
266070,116.03,116.05,116.05,116.02,430486.4,3710,2025-10-13 21:36:00,2025-10-13 21:36:59
266071,116.02,115.94,116.02,115.85,10648708.5,91870,2025-10-13 21:37:00,2025-10-13 21:37:59
266072,115.93,115.93,115.93,115.92,497339.5,4290,2025-10-13 21:38:00,2025-10-13 21:38:59
